In [ ]:
%stop_session

In [ ]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore",
  "--enable-glue-datacatalog": "true"
}

In [ ]:
#Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *
from delta.tables import DeltaTable
from datetime import *
from pyspark.sql.types import *
from pyspark.sql.utils import AnalysisException

BASE = "s3://data-lake-case-hotmart"

# D-1 
RUN_DATE = (date.today() - timedelta(days=1)).isoformat() #D-1 troque para a data que deseja navegar
# uat : RUN_DATE = "2023-03-31"   # teste também: "2023-01-23", "2023-02-28", "2023-07-15"

PATH_BRONZE_PURCHASE         = f"{BASE}/bronze/purchase"
PATH_BRONZE_PRODUCT_ITEM     = f"{BASE}/bronze/product_item"
PATH_BRONZE_EXTRA_INFO       = f"{BASE}/bronze/purchase_extra_info"

PATH_SILVER_PURCHASE         = f"{BASE}/silver/purchase_daily"
PATH_SILVER_PRODUCT_ITEM     = f"{BASE}/silver/product_item_daily"
PATH_SILVER_EXTRA_INFO       = f"{BASE}/silver/purchase_extra_info_daily"

PATH_GOLD                    = f"{BASE}/gold/gmv_daily"
PATH_GOLD_GMV                = f"{BASE}/gold/gmv_daily_by_subsidiary"

### silver_extra_info

In [ ]:
# 1) Ler Silver atual (se existir)
try:
    df_silver_current = spark.read.format("delta").load(PATH_SILVER_EXTRA_INFO)
except AnalysisException:
    df_silver_current = None

# 2) Última ingestion processada
last_ingestion = (
    df_silver_current.agg(max("bronze_ingestion_date").alias("max_date")).first()["max_date"]
) if df_silver_current is not None else None

# 3) Ler Bronze (incremental ou full)
df_bronze = spark.read.format("delta").load(PATH_BRONZE_EXTRA_INFO)

df_bronze_incremental = (
    df_bronze.filter(col("ingestion_date") > last_ingestion)
    if last_ingestion
    else df_bronze
)

# 4) Transformações Bronze → Silver
df_bronze_ready = (
    df_bronze_incremental
    .withColumn("line_created_at", to_utc_timestamp(current_timestamp(), "UTC"))
    .withColumnRenamed("ingestion_date", "bronze_ingestion_date")
)

# 4.1) Enriquecer purchase_partition (diagrama) a partir do purchase corrente, se existir
try:
    df_purchase_silver = spark.read.format("delta").load(PATH_SILVER_PURCHASE)
    df_purchase_current = df_purchase_silver.filter(col("is_current_record") == True) \
        .select("purchase_id", "purchase_partition")
    df_bronze_ready = df_bronze_ready.join(df_purchase_current, "purchase_id", "left")
except AnalysisException:
    pass

# 5) Union Silver + Bronze incremental
if df_silver_current is not None:
    cols_drop = [c for c in ["rn", "rk", "is_latest", "current_snapshot"] if c in df_silver_current.columns]
    df_union = df_silver_current.drop(*cols_drop).unionByName(df_bronze_ready, allowMissingColumns=True)
else:
    df_union = df_bronze_ready

# 6) Deduplicação por evento (CDC)
event_window = Window.partitionBy("purchase_id", "transaction_datetime").orderBy(col("bronze_ingestion_date").desc())

df_dedup = (
    df_union
    .withColumn("rn", row_number().over(event_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

# 7) Registro corrente por purchase_id (renomeado)
purchase_window = Window.partitionBy("purchase_id").orderBy(col("transaction_datetime").desc())

df_silver_extra_info_final = (
    df_dedup
    .withColumn("rk", row_number().over(purchase_window))
    .withColumn("Is_current_record", col("rk") == 1)  # renomeado (antes: is_latest)
    .drop("rk")
)

# 8) Escrita final (rebuild) - DELTA
df_silver_extra_info_final.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .save(PATH_SILVER_EXTRA_INFO)

In [ ]:
df_silver_extra_info_final.createOrReplaceTempView("vw_purchase_extra_info_silver")
spark.sql("SELECT * FROM vw_purchase_extra_info_silver ORDER BY purchase_id, transaction_datetime").show()